In [16]:
import warnings

warnings.simplefilter("ignore", DeprecationWarning)
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client.*")

In [17]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files[:20]:
        print(os.path.join(root, f))

/kaggle/input/datasets/ulrikthygepedersen/online-retail-dataset/online_retail.csv


In [18]:
import warnings

warnings.simplefilter("ignore", DeprecationWarning)
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client.*")

In [19]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/ulrikthygepedersen/online-retail-dataset/online_retail.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [20]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/ulrikthygepedersen/online-retail-dataset/online_retail.csv")

row_count = len(df)
basket_count_raw = df["InvoiceNo"].nunique()

df_valid = df.dropna(subset=["Description"]).copy()
basket_count_valid = df_valid["InvoiceNo"].nunique()

print("Eilučių skaičius:", row_count)
print("Unikalių InvoiceNo (raw krepšeliai):", basket_count_raw)
print("Validžių krepšelių su ne-null Description:", basket_count_valid)

Eilučių skaičius: 541909
Unikalių InvoiceNo (raw krepšeliai): 25900
Validžių krepšelių su ne-null Description: 24446


In [21]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. Nuskaitymas
df = pd.read_csv("/kaggle/input/datasets/ulrikthygepedersen/online-retail-dataset/online_retail.csv")

# 2. Paliekam tik eilutes su prekės pavadinimu
df_valid = df.dropna(subset=["Description"]).copy()

# 3. Formuojam krepšelius: vienas InvoiceNo = vienas basket
baskets = (
    df_valid.groupby("InvoiceNo")["Description"]
    .agg(lambda x: list(set(x)))
    .tolist()
)

print("Krepšelių skaičius FP-Growth analizei:", len(baskets))

# 4. One-hot kodavimas
te = TransactionEncoder()
te_ary = te.fit(baskets).transform(baskets)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

# 5. Funkcija skaičiavimui
def run_fp(min_support, min_confidence):
    freq = fpgrowth(basket_df, min_support=min_support, use_colnames=True)
    rules = association_rules(freq, metric="confidence", min_threshold=min_confidence)
    return len(freq), len(rules)

cases = [
    (0.03, 0.30),
    (0.02, 0.30),
    (0.03, 0.50),
]

for sup, conf in cases:
    freq_count, rules_count = run_fp(sup, conf)
    print(f"support={sup}, confidence={conf} -> frequent itemsets={freq_count}, rules={rules_count}")

Krepšelių skaičius FP-Growth analizei: 24446
support=0.03, confidence=0.3 -> frequent itemsets=86, rules=4
support=0.02, confidence=0.3 -> frequent itemsets=241, rules=75
support=0.03, confidence=0.5 -> frequent itemsets=86, rules=3


In [22]:
!pip -q install mlxtend

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

df = pd.read_csv("/kaggle/input/datasets/ulrikthygepedersen/online-retail-dataset/online_retail.csv")

def build_baskets(data):
    return (
        data.groupby("InvoiceNo")["Description"]
        .agg(lambda x: list(set(x)))
        .tolist()
    )

def run_fp(data, min_support, min_confidence):
    baskets = build_baskets(data)

    te = TransactionEncoder()
    te_ary = te.fit(baskets).transform(baskets)
    basket_df = pd.DataFrame(te_ary, columns=te.columns_)

    freq = fpgrowth(basket_df, min_support=min_support, use_colnames=True)
    rules = association_rules(freq, metric="confidence", min_threshold=min_confidence)

    return baskets, freq, rules

raw_df = df[df["Description"].notna()].copy()
clean_df = df[(df["Description"].notna()) & (df["Quantity"] > 0)].copy()

print("Visas df:", df.shape)
print("raw_df (tik su Description):", raw_df.shape)
print("clean_df (be grąžinimų ir be tuščių Description):", clean_df.shape)

Visas df: (541909, 8)
raw_df (tik su Description): (540455, 8)
clean_df (be grąžinimų ir be tuščių Description): (530693, 8)


In [23]:
raw_baskets_03_03, raw_freq_03_03, raw_rules_03_03 = run_fp(raw_df, 0.03, 0.30)
raw_baskets_02_03, raw_freq_02_03, raw_rules_02_03 = run_fp(raw_df, 0.02, 0.30)
raw_baskets_03_05, raw_freq_03_05, raw_rules_03_05 = run_fp(raw_df, 0.03, 0.50)

print("3% / 30% ->", len(raw_freq_03_03), "dažni rinkiniai,", len(raw_rules_03_03), "taisyklės")
print("2% / 30% ->", len(raw_freq_02_03), "dažni rinkiniai,", len(raw_rules_02_03), "taisyklės")
print("3% / 50% ->", len(raw_freq_03_05), "dažni rinkiniai,", len(raw_rules_03_05), "taisyklės")

3% / 30% -> 86 dažni rinkiniai, 4 taisyklės
2% / 30% -> 241 dažni rinkiniai, 75 taisyklės
3% / 50% -> 86 dažni rinkiniai, 3 taisyklės


In [24]:
# Naudojam jau suskaičiuotus raw_freq_02_03 ir raw_rules_02_03

singletons = raw_freq_02_03[raw_freq_02_03["itemsets"].apply(len) == 1].copy()
singletons["item"] = singletons["itemsets"].apply(lambda x: list(x)[0])
singletons["count"] = (singletons["support"] * len(raw_baskets_02_03)).round().astype(int)

top2 = singletons.sort_values("support", ascending=False).head(2).reset_index(drop=True)

top_item = top2.loc[0, "item"]
top_count = top2.loc[0, "count"]

second_item = top2.loc[1, "item"]
second_count = top2.loc[1, "count"]

ratio = top_count / second_count

print("TOP 2 prekės:")
print(top2[["item", "count", "support"]])
print()
print("Santykis:", round(ratio, 2))

rules_single = raw_rules_02_03[
    (raw_rules_02_03["antecedents"].apply(len) == 1) &
    (raw_rules_02_03["consequents"].apply(len) == 1)
].copy()

rules_single["antecedent"] = rules_single["antecedents"].apply(lambda x: list(x)[0])
rules_single["consequent"] = rules_single["consequents"].apply(lambda x: list(x)[0])

candidate = rules_single[
    rules_single["confidence"].between(0.54, 0.56)
].sort_values("lift", ascending=False)

print("\nKandidatės taisyklės apie 55% confidence:")
print(candidate[["antecedent", "consequent", "support", "confidence", "lift"]].head(10))

TOP 2 prekės:
                                 item  count   support
0  WHITE HANGING HEART T-LIGHT HOLDER   2302  0.094167
1            REGENCY CAKESTAND 3 TIER   2169  0.088726

Santykis: 1.06

Kandidatės taisyklės apie 55% confidence:
                          antecedent                         consequent  \
58  ROSES REGENCY TEACUP AND SAUCER      PINK REGENCY TEACUP AND SAUCER   
3        WOODEN FRAME ANTIQUE WHITE   WOODEN PICTURE FRAME WHITE FINISH   
29           LUNCH BAG PINK POLKADOT            LUNCH BAG RED RETROSPOT   
72                  JUMBO BAG APPLES            JUMBO BAG RED RETROSPOT   

     support  confidence       lift  
58  0.025117    0.548214  16.731144  
3   0.022294    0.552178  12.073838  
29  0.025076    0.552252   8.400970  
72  0.022703    0.555556   6.361176  


In [25]:
returned_count = (df["Quantity"] < 0).sum()
missing_desc_count = df["Description"].isna().sum()
removed_total = ((df["Quantity"] < 0) | (df["Description"].isna())).sum()

clean_df = df[(df["Quantity"] > 0) & (df["Description"].notna())].copy()

clean_rows = len(clean_df)
clean_baskets = clean_df["InvoiceNo"].nunique()

print("Grąžintų įrašų:", returned_count)
print("Įrašų be Description:", missing_desc_count)
print("Iš viso pašalinta eilučių:", removed_total)
print("Išvalyto rinkinio eilučių:", clean_rows)
print("Išvalyto rinkinio krepšelių:", clean_baskets)

Grąžintų įrašų: 10624
Įrašų be Description: 1454
Iš viso pašalinta eilučių: 11216
Išvalyto rinkinio eilučių: 530693
Išvalyto rinkinio krepšelių: 20136


In [26]:
clean_baskets_02_03, clean_freq_02_03, clean_rules_02_03 = run_fp(clean_df, 0.02, 0.30)
clean_baskets_03_03, clean_freq_03_03, clean_rules_03_03 = run_fp(clean_df, 0.03, 0.30)

print("Išvalytas rinkinys, 2% / 30% ->",
      len(clean_freq_02_03), "dažni rinkiniai,",
      len(clean_rules_02_03), "taisyklės")

print("Išvalytas rinkinys, 3% / 30% ->",
      len(clean_freq_03_03), "dažni rinkiniai,",
      len(clean_rules_03_03), "taisyklės")

Išvalytas rinkinys, 2% / 30% -> 375 dažni rinkiniai, 151 taisyklės
Išvalytas rinkinys, 3% / 30% -> 136 dažni rinkiniai, 16 taisyklės


In [27]:
!pip -q install mlxtend

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

df = pd.read_csv("/kaggle/input/datasets/ulrikthygepedersen/online-retail-dataset/online_retail.csv")

clean_df = df[(df["Description"].notna()) & (df["Quantity"] > 0)].copy()

def build_baskets(data):
    return (
        data.groupby("InvoiceNo")["Description"]
        .agg(lambda x: list(set(x)))
        .tolist()
    )

def run_fp(data, min_support, min_confidence):
    baskets = build_baskets(data)

    te = TransactionEncoder()
    te_ary = te.fit(baskets).transform(baskets)
    basket_df = pd.DataFrame(te_ary, columns=te.columns_)

    freq = fpgrowth(basket_df, min_support=min_support, use_colnames=True)
    rules = association_rules(freq, metric="confidence", min_threshold=min_confidence)

    return baskets, freq, rules

# ankstesni rezultatai palyginimui
raw_df = df[df["Description"].notna()].copy()

raw_baskets_02_03, raw_freq_02_03, raw_rules_02_03 = run_fp(raw_df, 0.02, 0.30)
raw_baskets_03_03, raw_freq_03_03, raw_rules_03_03 = run_fp(raw_df, 0.03, 0.30)

clean_baskets_02_03, clean_freq_02_03, clean_rules_02_03 = run_fp(clean_df, 0.02, 0.30)
clean_baskets_03_03, clean_freq_03_03, clean_rules_03_03 = run_fp(clean_df, 0.03, 0.30)

print("RAW 2%/30%:", len(raw_freq_02_03), len(raw_rules_02_03))
print("RAW 3%/30%:", len(raw_freq_03_03), len(raw_rules_03_03))
print("CLEAN 2%/30%:", len(clean_freq_02_03), len(clean_rules_02_03))
print("CLEAN 3%/30%:", len(clean_freq_03_03), len(clean_rules_03_03))

RAW 2%/30%: 241 75
RAW 3%/30%: 86 4
CLEAN 2%/30%: 375 151
CLEAN 3%/30%: 136 16


In [28]:
def itemset_size_counts(freq_df):
    temp = freq_df.copy()
    temp["size"] = temp["itemsets"].apply(len)
    return temp.groupby("size").size().to_dict()

counts_02 = itemset_size_counts(clean_freq_02_03)
counts_03 = itemset_size_counts(clean_freq_03_03)

print("Support 2%:", counts_02)
print("Support 3%:", counts_03)

Support 2%: {1: 293, 2: 79, 3: 3}
Support 3%: {1: 128, 2: 8}


In [29]:
singletons = clean_freq_02_03[clean_freq_02_03["itemsets"].apply(len) == 1].copy()
singletons["item"] = singletons["itemsets"].apply(lambda x: list(x)[0])
singletons["count"] = (singletons["support"] * len(clean_baskets_02_03)).round().astype(int)

top3 = singletons.sort_values("support", ascending=False).head(3).reset_index(drop=True)
print("TOP 3 prekės:")
print(top3[["item", "count", "support"]])

top_item = top3.loc[0, "item"]
top_count = top3.loc[0, "count"]

third_item = top3.loc[2, "item"]
third_count = top3.loc[2, "count"]

ratio = top_count / third_count
print("\nTop / third santykis:", round(ratio, 2))

rules_tmp = clean_rules_02_03.copy()
rules_tmp["ante"] = rules_tmp["antecedents"].apply(lambda x: list(x)[0] if len(x)==1 else ", ".join(sorted(list(x))))
rules_tmp["cons"] = rules_tmp["consequents"].apply(lambda x: list(x)[0] if len(x)==1 else ", ".join(sorted(list(x))))

candidate = rules_tmp[
    (rules_tmp["confidence"].between(0.71, 0.73)) &
    (rules_tmp["lift"].between(15.8, 16.0))
].sort_values("lift", ascending=False)

print("\nKandidatė taisyklė:")
print(candidate[["ante", "cons", "support", "confidence", "lift"]].head(10))

TOP 3 prekės:
                                 item  count   support
0  WHITE HANGING HEART T-LIGHT HOLDER   2260  0.112237
1             JUMBO BAG RED RETROSPOT   2092  0.103894
2            REGENCY CAKESTAND 3 TIER   1989  0.098778

Top / third santykis: 1.14

Kandidatė taisyklė:
                                   ante                               cons  \
150  GARDENERS KNEELING PAD CUP OF TEA   GARDENERS KNEELING PAD KEEP CALM    

      support  confidence       lift  
150  0.027116    0.720317  15.886413  
